# Simulation of MYC Transcription Factor Network

This notebook simulates data and applies causal inference to a transcription 
factor network pulled from INDRA. The network is focused around the TF MYC.

The network was pulled such that there is interesting graphical topology that 
affects the analysis. To repull the network, see the `single_node.py` file.

In [ ]:
# Load packages
from MScausality.causal_model.LVM import LVM
from MScausality.simulation.simulation import simulate_data
from MScausality.data_analysis.normalization import normalize

from MScausality.data_analysis.dataProcess import dataProcess

import pyro
import pandas as pd
import numpy as np
from sklearn import linear_model

import networkx as nx
import y0
from y0.algorithm.simplify_latent import simplify_latent_dag
from y0.algorithm.identify import Identification, identify
from y0.dsl import P, Variable

import pickle
import copy

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## Construct network

In [ ]:
    # network = build_network(["MYC"], "gene", client, 
    #                         evidence_count=[85, 80, 70,50], levels=4,
    #                         upstream=True, downstream=[True, False, False, False])

In [ ]:
def build_tf_network():
    """
    Create TF MYC graph in networkx
    
    """
    graph = nx.DiGraph()

    ## Add edges
    graph.add_edge("LEP", "IL6")
    graph.add_edge("LEP", "STAT3")
    graph.add_edge("IL6", "STAT3")
    graph.add_edge("LEP", "TNP")
    graph.add_edge("STAT3", "MYC")
    graph.add_edge("WNT3A", "CTNNB1")
    graph.add_edge("CTNNB1", "MYC")
    graph.add_edge("TNP", "CDKN1A")
    graph.add_edge("MYC", "CDKN1A")
    graph.add_edge("MYC", "TERT")
    
    return graph

def build_admg(graph):

    """

    Creates acyclic directed mixed graph (ADMG) from networkx graph with latent variables.
    
    """

    ## Define obs vs latent nodes
    all_nodes = ["LEP", "IL6", "STAT3", "TNP", "MYC", 
                 "WNT3A", "CTNNB1", "CDKN1A", "TERT"]
    obs_nodes = ["IL6", "STAT3", "MYC", "CTNNB1", "CDKN1A", "TERT"]
            
    attrs = {node: (True if node not in obs_nodes and 
                    node != "\\n" else False) for node in all_nodes}

    nx.set_node_attributes(graph, attrs, name="hidden")
    ## Use y0 to build ADMG
    mapping = dict(zip(list(graph.nodes), 
                      [Variable(i) for i in list(graph.nodes)]))
    graph = nx.relabel_nodes(graph, mapping)
    
    ## Use y0 to build ADMG
    simplified_graph = simplify_latent_dag(graph.copy(), tag="hidden")
    y0_graph = y0.graph.NxMixedGraph()
    y0_graph = y0_graph.from_latent_variable_dag(simplified_graph.graph, "hidden")
    
    return y0_graph

In [ ]:
graph = build_tf_network()
y0_graph = build_admg(graph)

In [ ]:
def plot_latent_graph(y0_graph, figure_size=(4, 3), title=None):

    ## Create new graph and specify color and shape of observed vs latent edges
    temp_g = nx.DiGraph()

    for d_edge in list(y0_graph.directed.edges):
        temp_g.add_edge(d_edge[0], d_edge[1], color="black", style='-', size=30)
    for u_edge in list(y0_graph.undirected.edges):
        if temp_g.has_edge(u_edge[0], u_edge[1]):
            temp_g.add_edge(u_edge[1], u_edge[0], color="red", style='--', size=1)
        else:
            temp_g.add_edge(u_edge[0], u_edge[1], color="red", style='--', size=1)

    ## Extract edge attributes
    pos = nx.random_layout(temp_g)
    edges = temp_g.edges()
    colors = [temp_g[u][v]['color'] for u, v in edges]
    styles = [temp_g[u][v]['style'] for u, v in edges]
    arrowsizes = [temp_g[u][v]['size'] for u, v in edges]

    ## Plot
    fig, ax = plt.subplots(figsize=figure_size)
    nx.draw_networkx_nodes(temp_g, pos=pos, node_size=1000, margins=[.1, .1], alpha=.7)
    nx.draw_networkx_labels(temp_g, pos=pos, font_weight='bold')
    nx.draw_networkx_edges(temp_g, pos=pos, ax=ax, connectionstyle='arc3, rad = 0.1',
                           edge_color=colors, width=3, style=styles, arrowsize=arrowsizes)
    if title is not None:
        ax.set_title(title)
    plt.show()
    
plot_latent_graph(y0_graph, figure_size=(12, 8))

In [ ]:
query = Identification.from_expression(graph=y0_graph, query=P(Variable("MYC") @ Variable("STAT3")))
estimand = identify(query)
estimand


## Ground truth invervention

Generate interventional data using the ground truth network for comparison.

In [ ]:
# ## Coefficients for relations
# myc_coef = {'LEP': {'intercept': 18., "error": 3},
#               'WNT3A': {'intercept': 17., "error": 3},
#               'IL6': {'intercept': 3, "error": 1, 
#                       'LEP': 0.6},
#               'CTNNB1': {'intercept': 5, "error": 1, 'WNT3A': .5},
#               'STAT3': {'intercept': 1.6, "error": 1, 
#                        'IL6': -0.5, 'LEP': 0.8},
#               'TNP': {'intercept': 2., "error": 1, 'LEP': 0.75},
#               'MYC': {'intercept': 2, "error": 1,
#                       'STAT3': 1., 'CTNNB1': -.2},
#               'CDKN1A': {'intercept': 3., "error": 1, 'MYC': 0.75, "TNP":.45},
#               'TERT': {'intercept': 4., "error": 1, 'MYC': 1.2}
#              }

## Coefficients for relations
myc_coef = {'LEP': {'intercept': 18., "error": .25},
              'WNT3A': {'intercept': 17., "error": .25},
              'IL6': {'intercept': 3, "error": .2, 
                      'LEP': 0.6},
              'CTNNB1': {'intercept': 5, "error": .2, 'WNT3A': .5},
              'STAT3': {'intercept': 1.6, "error": .2, 
                       'IL6': -0.5, 'LEP': 0.8},
              'TNP': {'intercept': 2., "error": .25, 'LEP': 0.75},
              'MYC': {'intercept': 2, "error": .25,
                      'STAT3': 1., 'CTNNB1': -.2},
              'CDKN1A': {'intercept': 3., "error": .25, 'MYC': 0.75, "TNP":.45},
              'TERT': {'intercept': 4., "error": .25, 'MYC': 1.2}
             }

In [ ]:
def obs_gt_sim(coef, n):
    
    """
    Observational ground truth simulation of network
    """
    
    samples = dict()

    for key in coef.keys():
        eq_vals = coef[key]

        if len(eq_vals) > 2:
            samples[key] = eq_vals["intercept"] + np.sum(
                [eq_vals[reg]*samples[reg] for reg in eq_vals.keys() 
                 if reg not in ["intercept", "error"]], axis=0) + np.random.normal(
                     0, eq_vals["error"], n)
        else:
            samples[key] = np.random.normal(eq_vals["intercept"], eq_vals["error"], n)

    return samples

In [ ]:
observational_data= obs_gt_sim(myc_coef, 100)
fig, ax = plt.subplots(3, 3, figsize=(12, 12))
t=0
for i in range(3):
    ax[0, i].hist(observational_data[list(observational_data.keys())[t]])
    ax[1, i].hist(observational_data[list(observational_data.keys())[t+1]])
    ax[2, i].hist(observational_data[list(observational_data.keys())[t+2]])

    ax[0, i].set_title(list(observational_data.keys())[t])
    ax[1, i].set_title(list(observational_data.keys())[t+1])
    ax[2, i].set_title(list(observational_data.keys())[t+2])

    ax[0, i].set_xlim(-5, 30)
    ax[1, i].set_xlim(-5, 30)
    ax[2, i].set_xlim(-5, 30)

    t+=3

In [ ]:
def int_gt_sim(coef, n, protein_int_name, protein_int_value):

    """
    Ras interventional simulation of network (Ras=10 vs Ras=20)
    """
    
    samples = dict()

    for key in coef.keys():
        if key != protein_int_name:
            eq_vals = coef[key]

            if len(eq_vals) > 2:
                samples[key] = eq_vals["intercept"] + np.sum(
                    [eq_vals[reg]*samples[reg] for reg in eq_vals.keys() 
                    if reg not in ["intercept", "error"]], axis=0) + np.random.normal(
                        0, eq_vals["error"], n)
            else:
                samples[key] = np.random.normal(eq_vals["intercept"], eq_vals["error"], n)
        else:
            samples[protein_int_name] = np.repeat(protein_int_value, n)

    return samples

gt_int_IL6_low = int_gt_sim(myc_coef, 100000, "IL6", 15)
gt_int_IL6_high = int_gt_sim(myc_coef, 100000, "IL6", 10)

In [ ]:
np.mean(gt_int_IL6_low["MYC"])

In [ ]:
np.mean(gt_int_IL6_high["MYC"])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"], bins=50, alpha=0.75, label="IL6=20")
ax.axvline((gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"]).mean(), color="darkred", linestyle="dashed", lw=4)
print("Ground Truth ACE: ", round(np.mean(gt_int_IL6_high["MYC"]) - np.mean(gt_int_IL6_low["MYC"]), 2))

ax.set_title("IL6 log FC = -2", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
ax.set_ylabel("")


ax.set_xlim(-5,5)
ax.set_yticklabels([])
ax.set_yticklabels([])

ax.xaxis.set_tick_params(labelsize=16)

## Run inference

In [ ]:
myc_sim = simulate_data(graph, coefficients=myc_coef, include_missing=True, 
                        mnar_missing_param=[-3, .4], cell_type=False, 
                        n_cells=3, n=100, seed=1)
myc_sim["Feature_data"].to_csv("../../data/MYC_pathway/feature_data_100_reps.csv", index=False)

In [ ]:
# myc_sim_protein_data = pd.read_csv("../../data/MYC_pathway/protein_data_100_reps.csv")
myc_sim_protein_data = dataProcess(myc_sim["Feature_data"], normalization=False, MBimpute=False, sim_data=True) 
myc_sim_protein_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

plot_data = myc_sim_protein_data.loc[:, ["STAT3", "MYC"]].dropna()
x = plot_data["STAT3"]
y = plot_data["MYC"]

ax.scatter(x, y)
m, b = np.polyfit(x.values, y.values, 1)
ax.plot(x, m*x + b, color="red", lw=4)

ax.set_title("Regression Analysis", size=24)
ax.set_xlabel("STAT3", size=24)
ax.set_ylabel("MYC", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

### Run model

In [ ]:
input_data

In [ ]:
transformed_data = normalize(myc_sim_protein_data, wide_format=True)
input_data = transformed_data["df"]
scale_metrics = transformed_data["adj_metrics"]


pyro.clear_param_store()
lvm = LVM(input_data, 
          y0_graph)
lvm.prepare_graph()
lvm.prepare_data()
lvm.get_priors()
lvm.fit_model(num_steps=4000,
              patience=100, min_delta=5)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

# ax.hist(np.array(first_int), bins=30, alpha=.75, label="STAT3=10")
# ax.axvline(np.mean(np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)
# sns.histplot(gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"], ax=ax2, 
#              kde=True, alpha=.01, color="grey",
#              label="Ground Truth")
ax.hist(myc_sim_protein_data["MYC"] - myc_sim_protein_data["MYC"].mean(), 
        bins=12, alpha=0.75, label="Observational Data", color="orange")
ax.axvline(0, color="darkred", linestyle="dashed", lw=4)

ax.legend(fontsize=16)
ax.set_title("STAT3 Observational Distribution", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
ax.set_ylabel("")


ax.set_xlim(-10,10)
ax.set_yticklabels([])
ax2.set_yticklabels([])

ax.xaxis.set_tick_params(labelsize=16)
# plt.yticks(fontsize=20)

# fig, ax = plt.subplots(figsize=(8, 6))
# ax.hist(, bins=50, alpha=0.5, label="IL6=10")
# print("Ground Truth ACE: ", round(np.mean(gt_int_IL6_high["MYC"]) - np.mean(gt_int_IL6_low["MYC"]), 2))

In [ ]:
lvm.intervention("STAT3", "MYC", 10)
first_int = lvm.intervention_samples

lvm.intervention("STAT3", "MYC", 12)
second_int = lvm.intervention_samples

intervention_results = [first_int, second_int, lvm]

fig, ax = plt.subplots(figsize=(12,6))

# ax.hist(np.array(first_int), bins=30, alpha=.75, label="STAT3=10")
# ax.axvline(np.mean(np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)
ax2 = ax.twinx()
# sns.histplot(gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"], ax=ax2, 
#              kde=True, alpha=.01, color="grey",
#              label="Ground Truth")
ax.hist(myc_sim_protein_data["MYC"] - myc_sim_protein_data["MYC"].mean(), 
        bins=10, alpha=0.75, label="Observational Data", color="orange")
ax.hist(0, 
        bins=10, alpha=0.75, label="Estimated Interventional Effect", color="blue")
ax2.axvline(0, color="darkred", linestyle="dashed", lw=4)

ax2.hist(np.array(first_int) - np.array(second_int), bins=25, alpha=.75, label="Estimated Interventional Effect")
ax2.axvline(np.mean(np.array(first_int) - np.array(second_int)), color="darkblue", linestyle="dashed", lw=4)

ax.legend(fontsize=14, loc="upper right")
ax.set_title("STAT3 Log Fold Change = -2", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
ax2.set_ylabel("")
ax.set_ylabel("")

ax.set_xlim(-10,10)
ax.set_yticklabels([])
ax2.set_yticklabels([])

ax.xaxis.set_tick_params(labelsize=16)

In [ ]:
lvm.intervention("STAT3", "MYC", 10)
first_int = lvm.intervention_samples

lvm.intervention("STAT3", "MYC", 12)
second_int = lvm.intervention_samples

intervention_results = [first_int, second_int, lvm]

fig, ax = plt.subplots(figsize=(12,6))

# ax.hist(np.array(first_int), bins=30, alpha=.75, label="STAT3=10")
# ax.axvline(np.mean(np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)
ax2 = ax.twinx()
sns.histplot(gt_int_IL6_low["MYC"]-gt_int_IL6_high["MYC"]+.5, ax=ax2, 
             kde=True, alpha=.25, color="grey",
             label="Ground Truth")
# ax.hist(myc_sim_protein_data["MYC"] - myc_sim_protein_data["MYC"].mean(), 
#         bins=10, alpha=0.75, label="Observational Data", color="orange")
ax.hist(0, 
        bins=10, alpha=0.75, label="Estimated Interventional Effect", color="blue")
# ax2.axvline(0, color="darkred", linestyle="dashed", lw=4)

ax.hist(np.array(first_int) - np.array(second_int), bins=25, alpha=.75, label="Estimated Interventional Effect")
ax.axvline(np.mean(np.array(first_int) - np.array(second_int)), color="darkblue", linestyle="dashed", lw=4)

ax.legend(fontsize=14, loc="upper right")
ax.set_title("STAT3 Log Fold Change = -2", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
ax2.set_ylabel("")
ax.set_ylabel("")

ax.set_xlim(-10,10)
ax.set_yticklabels([])
ax2.set_yticklabels([])

ax.xaxis.set_tick_params(labelsize=16)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.hist(np.array(first_int) - np.array(second_int), color="darkblue",
        bins=30, alpha=.65, label="STAT3 Log FC = -2")
ax.axvline(np.mean(np.array(first_int) - np.array(second_int)), color="darkred", linestyle="dashed", lw=4)

# ax.axvline(np.mean(np.array(second_int) - np.array(first_int)), color="darkred", linestyle="dashed", lw=1)

# ax.axvline(np.mean(np.array(gt_int_IL6_high["MYC"]) - np.array(gt_int_IL6_low["MYC"])), 
#            color="darkblue", linestyle="dashed", lw=1)

ax.legend(fontsize=20)
ax.set_title("STAT3 Intervention", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
# ax.set_ylabel("MYC", size=24)
plt.xticks(fontsize=16)
ax.set_yticklabels([])
# ax2.set_yticks(fontsize=0)

ax.set_xlim(-10,10)


In [ ]:
lvm.intervention("CTNNB1", "MYC", 10)
first_int = lvm.intervention_samples

lvm.intervention("CTNNB1", "MYC", 12)
second_int = lvm.intervention_samples

intervention_results = [first_int, second_int, lvm]

fig, ax = plt.subplots(figsize=(12, 7))

# ax.hist(np.array(first_int), bins=30, alpha=.75, label="STAT3=10")
# ax.axvline(np.mean(np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)
ax.hist(np.array(first_int) - np.array(second_int), bins=25, color="darkorange", alpha=.75, label="CTNNB1 Log FC = -2")
ax.axvline(np.mean(np.array(first_int) - np.array(second_int)), color="darkred", linestyle="dashed", lw=4)

ax.legend(fontsize=20)
ax.set_title("CTNNB1 Intervention", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
# ax.set_ylabel("MYC", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
ax.set_yticklabels([])

ax.set_xlim(-10,10)

# fig, ax = plt.subplots(figsize=(8, 6))
# ax.hist(, bins=50, alpha=0.5, label="IL6=10")
# ax2 = ax.twinx()
# ax2.hist(gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"], bins=50, alpha=0.5, label="IL6=20")
# print("Ground Truth ACE: ", round(np.mean(gt_int_IL6_high["MYC"]) - np.mean(gt_int_IL6_low["MYC"]), 2))

In [ ]:
lvm.intervention("IL6", "MYC", 10)
first_int = lvm.intervention_samples

lvm.intervention("IL6", "MYC", 12)
second_int = lvm.intervention_samples

intervention_results = [first_int, second_int, lvm]

fig, ax = plt.subplots(figsize=(12,6))

# ax.hist(np.array(first_int), bins=30, alpha=.75, label="STAT3=10")
# ax.axvline(np.mean(np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)
ax2 = ax.twinx()
# sns.histplot(gt_int_IL6_high["MYC"] - gt_int_IL6_low["MYC"], ax=ax2, 
#              kde=True, alpha=.01, color="grey",
#              label="Ground Truth")
ax.hist(myc_sim_protein_data["MYC"] - myc_sim_protein_data["MYC"].mean(), 
        bins=10, alpha=0.75, label="Observational Data", color="orange")
ax.hist(0, 
        bins=10, alpha=0.75, label="Estimated Interventional Effect", color="blue")
ax2.axvline(0, color="darkred", linestyle="dashed", lw=4)

ax2.hist(np.array(second_int) - np.array(first_int), bins=25, alpha=.75, label="Estimated Interventional Effect")
ax2.axvline(np.mean(np.array(second_int) - np.array(first_int)), color="darkblue", linestyle="dashed", lw=4)

ax.legend(fontsize=14, loc="upper left")
ax.set_title("IL6 Log Fold Change = 2", size=24)
ax.set_xlabel("MYC - Log Fold Change", size=24)
ax2.set_ylabel("")
ax.set_ylabel("")

ax.set_xlim(-10,10)
ax.set_yticklabels([])
ax2.set_yticklabels([])

ax.xaxis.set_tick_params(labelsize=16)

## Missing value imputation

In [ ]:
high_missing_data = simulate_data(graph, coefficients=myc_coef, 
                                  include_missing=True, 
                                  mar_missing_param=.1, mnar_missing_param=[-7, .4], 
                                  n=300, seed=2)
high_missing_data["Feature_data"].to_csv("../../data/MYC_pathway/high_missing_feature_data.csv", index=False)

In [ ]:
high_missing_protein_data = pd.read_csv("../../data/MYC_pathway/high_missing_protein_data.csv")
high_missing_protein_data.head()

In [ ]:
high_missing_protein_data.loc[((high_missing_protein_data["MYC"] < 12) & (-np.isnan(high_missing_protein_data["STAT3"]))), "MYC"] = np.nan

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

plot_data = high_missing_protein_data.loc[:, ["MYC", "STAT3"]].dropna()
x = plot_data["STAT3"]
y = plot_data["MYC"]

ax.scatter(x, y)
m, b = np.polyfit(x.values, y.values, 1)
ax.plot(x, m*x + b, color="red", lw=4)

ax.set_title("Regression Analysis", size=24)
ax.set_xlabel("STAT3", size=24)
ax.set_ylabel("MYC", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,6))

ax[0].bar(x = myc_sim_protein_data.iloc[: ,1:].isnull().sum(axis = 0).index,
          height=myc_sim_protein_data.iloc[: ,1:].isnull().sum(axis = 0) / len(myc_sim_protein_data))

ax[1].bar(x = high_missing_protein_data.iloc[: ,1:].isnull().sum(axis = 0).index, 
       height=high_missing_protein_data.iloc[: ,1:].isnull().sum(axis = 0) / len(high_missing_protein_data))

ax[0].set_ylim(0,1)
ax[1].set_ylim(0,1)

ax[0].set_title("Low Missingness", size=20)
ax[1].set_title("High Missingness", size=20)

ax[0].set_xlabel("Gene", size=20)
ax[1].set_xlabel("Gene", size=20)

ax[0].set_ylabel("Percent Missing Replicates", size=20)
ax[1].set_ylabel("Percent Missing Replicates", size=20)

In [ ]:
pyro.clear_param_store()
high_missing_lvm = LVM(
    high_missing_protein_data.iloc[:,1:], 
    y0_graph)
high_missing_lvm.prepare_graph()
high_missing_lvm.prepare_data()
high_missing_lvm.fit_model(num_steps=4000)

high_missing_lvm.intervention("STAT3", "MYC", 10)
first_int = high_missing_lvm.intervention_samples

high_missing_lvm.intervention("STAT3", "MYC", 20)
second_int = high_missing_lvm.intervention_samples

high_missing_intervention_results = [first_int, second_int, high_missing_lvm]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,6))

ax[0].hist(intervention_results[0], label="Ras=10", alpha=.75, bins=25)
ax[0].hist(intervention_results[1], label="Ras=20", alpha=.75, bins=25)
ax[0].axvline(x=intervention_results[0].mean(), color="Blue", lw=2, linestyle="--")
ax[0].axvline(x=intervention_results[1].mean(), color="Red", lw=2, linestyle="--")
ax[0].legend()

ax[1].hist(high_missing_intervention_results[0], label="Ras=10", alpha=.75, bins=25)
ax[1].hist(high_missing_intervention_results[1], label="Ras=20", alpha=.75, bins=25)
ax[1].axvline(x=high_missing_intervention_results[0].mean(), color="Blue", lw=2, linestyle="--")
ax[1].axvline(x=high_missing_intervention_results[1].mean(), color="Red", lw=2, linestyle="--")
ax[1].legend()

ax[0].set_xlim(-10,40)
ax[1].set_xlim(-10,40)

plt.legend()

ax[0].set_title("Low Missing Interventional Distribution", size=20)
ax[0].set_xlabel("MYC - Log Intensity", size=20)

ax[1].set_title("High Missing Interventional Distribution", size=20)
ax[1].set_xlabel("MYC - Log Intensity", size=20)

# # ax.set_ylabel("Erk", size=24)
# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)

### Compare imputed values

In [ ]:
def format_int_data(df, p1, p2):
    df.loc[:, "full_int"] = np.where(np.isnan(df["imp_mean"]),
                                        df["intensity"], df["imp_mean"])

    p1_df = df.loc[df["protein"] == p1, ["intensity", "full_int"]]
    p1_df.loc[:, "imp_{0}".format(p1)] = np.isnan(p1_df["intensity"])
    p1_df.loc[:, p1] = p1_df.loc[:, "intensity"]
    p1_df.loc[:, "full_{0}".format(p1)] = p1_df.loc[:, "full_int"]

    p2_df = df.loc[df["protein"] == p2, ["intensity", "full_int"]]
    p2_df.loc[:, "imp_{0}".format(p2)] = np.isnan(p2_df["intensity"])
    p2_df.loc[:, p2] = p2_df.loc[:, "intensity"]
    p2_df.loc[:, "full_{0}".format(p2)] = p2_df.loc[:, "full_int"]


    df = pd.concat([p1_df.loc[:, [p1, 
                                  "full_{0}".format(p1),
                                  "imp_{0}".format(p1)]].reset_index(drop=True), 
                    p2_df.loc[:, [p2, 
                                  "full_{0}".format(p2),
                                  "imp_{0}".format(p2)]].reset_index(drop=True)], 
                    axis=1)
    return df

high_missing_plot_df = format_int_data(high_missing_intervention_results[2].imputed_data, "MYC", "STAT3")
low_missing_plot_df = format_int_data(intervention_results[2].imputed_data, "MYC", "STAT3")

In [ ]:
low_missing_lm = linear_model.LinearRegression()
x = low_missing_plot_df.dropna()[["STAT3"]]
y = low_missing_plot_df.dropna()[["MYC"]]
low_missing_lm.fit(x, y)

low_missing_imp_lm = linear_model.LinearRegression()
x = low_missing_plot_df[["full_STAT3"]]
y = low_missing_plot_df[["full_MYC"]]
low_missing_imp_lm.fit(x, y)

high_missing_lm = linear_model.LinearRegression()
x = high_missing_plot_df.dropna()[["STAT3"]]
y = high_missing_plot_df.dropna()[["MYC"]]
high_missing_lm.fit(x, y)

high_missing_imp_lm = linear_model.LinearRegression()
x = high_missing_plot_df[["full_STAT3"]]
y = high_missing_plot_df[["full_MYC"]]
high_missing_imp_lm.fit(x, y)

In [ ]:
## Plot
fig, ax = plt.subplots(2, 2, figsize=(14,8))

colors1 = np.where((low_missing_plot_df["imp_MYC"] == True) & \
                   (low_missing_plot_df["imp_STAT3"] == True), "Red", 
                  np.where((low_missing_plot_df["imp_MYC"] == True) | \
                           (low_missing_plot_df["imp_STAT3"] == True), "Orange", "Blue"))

colors2 = np.where((high_missing_plot_df["imp_MYC"] == True) & \
                   (high_missing_plot_df["imp_STAT3"] == True), "Red", 
                  np.where((high_missing_plot_df["imp_MYC"] == True) | \
                           (high_missing_plot_df["imp_STAT3"] == True), "Orange", "Blue"))

ax[0,0].scatter(low_missing_plot_df["STAT3"], low_missing_plot_df["MYC"], c=colors1, alpha=.6)
ax[0,0].plot(low_missing_plot_df["STAT3"], 
           low_missing_lm.coef_[0]*low_missing_plot_df["STAT3"] + low_missing_lm.intercept_[0], 
           color="black", lw=4)

ax[1,0].scatter(low_missing_plot_df["full_STAT3"], low_missing_plot_df["full_MYC"], 
              c=colors1, alpha=.6)
ax[1,0].plot(low_missing_plot_df["full_STAT3"], 
           low_missing_imp_lm.coef_[0]*low_missing_plot_df["full_STAT3"] + low_missing_imp_lm.intercept_[0], 
           color="black", lw=4)

ax[0,1].scatter(high_missing_plot_df["STAT3"], high_missing_plot_df["MYC"], 
              c=colors2, alpha=.6)
ax[0,1].plot(np.arange(5,17), 
           high_missing_lm.coef_[0]*np.arange(5,17) + high_missing_lm.intercept_[0], 
           color="black", lw=4)

ax[1,1].scatter(high_missing_plot_df["full_STAT3"], high_missing_plot_df["full_MYC"], 
              c=colors2, alpha=.6)
ax[1,1].plot(high_missing_plot_df["full_STAT3"], 
           high_missing_imp_lm.coef_[0]*high_missing_plot_df["full_STAT3"] + high_missing_imp_lm.intercept_[0], 
           color="black", lw=4)

red_patch = mpatches.Patch(color='red', label='Both Missing')
orange_patch = mpatches.Patch(color='orange', label='One Missing')
blue_patch = mpatches.Patch(color='blue', label='Observed')

ax[1,1].legend(handles=[red_patch, orange_patch, blue_patch])
ax[1,0].legend(handles=[red_patch, orange_patch, blue_patch])

ax[0,0].set_xlim(0,20)
ax[1,1].set_xlim(0,20)
ax[0,1].set_xlim(0,20)
ax[1,0].set_xlim(0,20)
ax[0,0].set_ylim(0,20)
ax[1,1].set_ylim(0,20)
ax[0,1].set_ylim(0,20)
ax[1,0].set_ylim(0,20)

ax[0,0].set_title("Low missingness - no imputation", size=20)
ax[1,1].set_title("High missingness - with imputation", size=20)
ax[1,0].set_title("Low missingness - with imputation", size=20)
ax[0,1].set_title("High missingness - no imputation", size=20)

ax[0,0].set_xlabel("")
ax[0,1].set_xlabel("")
ax[1,1].set_xlabel("STAT3", size=20)
ax[1,0].set_xlabel("STAT3", size=20)

ax[0,0].set_ylabel("MYC", size=20)
ax[0,1].set_ylabel("MYC", size=20)
ax[1,1].set_ylabel("MYC", size=20)
ax[1,0].set_ylabel("MYC", size=20)

## Test other ML methods

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_validate

In [ ]:
myc_sim_protein_data.head()

In [ ]:
complete_set = myc_sim_protein_data.dropna()
X = complete_set.loc[:,["CDKN1A", "CTNNB1", "IL6", 
                                "LEP", "STAT3", 
                                "TERT", "TNP", "WNT3A"]]
y = complete_set["MYC"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

### Linear Regression

In [ ]:
lm = linear_model.LinearRegression()
lm.fit(X_train, y_train)

In [ ]:
dict(
    zip(["CDKN1A", "CTNNB1", "IL6", 
         "LEP", "STAT3", 
         "TERT", "TNP", "WNT3A"], 
     
    lm.coef_)
)

### Random Forest

In [ ]:
rf = RandomForestRegressor()
cv_results = cross_validate(rf, X_train, y_train, cv=5)

In [ ]:
cv_results

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
np.sqrt(np.mean((y_test - rf.predict(X_test))**2))

In [ ]:
dict(
    zip(["CDKN1A", "CTNNB1", "IL6", 
         "LEP", "STAT3", 
         "TERT", "TNP", "WNT3A"], 
         rf.feature_importances_)
    )

In [ ]:
X_test.columns

In [ ]:
test=X_test
test.loc[:, "TERT"] = 20
pd.Series(rf.predict(test)).hist()

In [ ]:
test = pd.DataFrame()

for col in X_test.columns:
    if col != "STAT3":
        test.loc[:, col] = np.random.uniform(0, 20, 10000)
    else:
        test.loc[:, "STAT3"] = np.repeat(10, 10000)

In [ ]:
len(rf.predict(test))

In [ ]:
pd.Series(rf.predict(test)).hist()

In [ ]:
test = pd.DataFrame()

for col in X_test.columns:
    if col != "STAT3":
        test.loc[:, col] = np.random.uniform(0, 20, 10000)
    else:
        test.loc[:, "STAT3"] = np.repeat(20, 10000)

pd.Series(rf.predict(test)).hist()

### XGBoost

In [ ]:
dtrain_reg = xgb.DMatrix(X_train, y_train)
params = {"objective": "reg:squarederror"}

n = 100
model = xgb.train(
   params=params,
   dtrain=dtrain_reg,
   num_boost_round=n,
)

In [ ]:
xgb.plot_importance(model)

### NN

In [ ]:
from sklearn.neural_network import MLPRegressor ## TODO: Replace with something better (but simple prob.. Keras?)

In [ ]:
myc_sim_high_rep = simulate_data(graph, coefficients=myc_coef, include_missing=True, 
                        mnar_missing_param=[-3, .4], cell_type=False, 
                        n_cells=3, n=10000, seed=1)
myc_sim_high_rep["Feature_data"].to_csv("../../data/MYC_pathway/high_rep_feature_data.csv", index=False)

In [ ]:
protein_data_high_rep = pd.read_csv("../../data/MYC_pathway/protein_data.csv")

In [ ]:
complete_set = protein_data_high_rep.dropna()
X = complete_set.loc[:,["CDKN1A", "CTNNB1", "IL6", 
                                "LEP", "STAT3", 
                                "TERT", "TNP", "WNT3A"]]
y = complete_set["MYC"]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.9, random_state=1)

In [ ]:
nn = MLPRegressor(random_state=1, max_iter=500).fit(X_train, y_train)
nn.predict(X_test[:])
np.sqrt(np.mean((y_test - nn.predict(X_test))**2))

In [ ]:
test=X_test.copy()
test.loc[:, "STAT3"] = 10
int0 = nn.predict(test)

test=X_test.copy()
test.loc[:, "STAT3"] = 20
int1 = nn.predict(test)

np.mean(int1) - np.mean(int0)

In [ ]:
pd.Series(int1).hist()

In [ ]:
pd.Series(int0).hist()

In [ ]:
test=X_test.copy()
test.loc[:, "TERT"] = 10
int0 = nn.predict(test)

test=X_test.copy()
test.loc[:, "TERT"] = 20
int1 = nn.predict(test)

np.mean(int1) - np.mean(int0)

In [ ]:
complete_set.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

plot_lm = linear_model.LinearRegression()
plot_lm.fit(complete_set["IL6"].values.reshape(-1, 1), complete_set["MYC"])

ax.scatter(complete_set["IL6"], complete_set["MYC"], color="navy", alpha=.75)
ax.plot(np.arange(8,19.5, .25), plot_lm.coef_[0]*np.arange(8,19.5, .25) + plot_lm.intercept_, 
        color="red", lw=5, alpha=.75)
ax.set_title("P(MYC | IL6)", size=24)
ax.set_xlabel("IL6", size=24)
ax.set_ylabel("MYC", size=24)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)

ax.set_xlim(7.5, 20.)
ax.set_ylim(4.5, 16.5)

In [ ]:
plot_x = complete_set.loc[complete_set["LEP"].between(18,19), "IL6"]
plot_y = complete_set.loc[complete_set["LEP"].between(18,19), "MYC"]

fig, ax = plt.subplots(figsize=(10,8))

plot_lm = linear_model.LinearRegression()
plot_lm.fit(plot_x.values.reshape(-1, 1), plot_y)

ax.scatter(plot_x, plot_y, color="navy", alpha=.75)
ax.plot(np.arange(11.6,17.5, .25), plot_lm.coef_[0]*np.arange(11.6,17.5, .25) + plot_lm.intercept_, 
        color="red", lw=5, alpha=.75)
ax.set_title("P(MYC | IL6, LEP)", size=24)
ax.set_xlabel("IL6", size=24)
ax.set_ylabel("MYC", size=24)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)

ax.set_xlim(11.5, 17.5)
ax.set_ylim(5.5, 13.5)

In [ ]:
bucket = [0, 14, 16, 18, 20, 22, np.inf]

plot_list = list()
lm_list = list()

for i in range(1, len(bucket)):

    temp_df = complete_set.loc[
        complete_set["LEP"].between(bucket[i-1], bucket[i]), 
        ["IL6", "MYC"]]
    
    temp_df.loc[:, "LEP"] = f"{bucket[i-1]}-{bucket[i]}"
    temp_df.loc[:, "test"] = i
    
    plot_list.append(temp_df)

    plot_lm = linear_model.LinearRegression()
    plot_lm.fit(temp_df["IL6"].values.reshape(-1, 1), temp_df["MYC"])
    lm_list.append(plot_lm)

plot_df = pd.concat(plot_list)
plot_df = plot_df[plot_df["test"].between(2,5)]


In [ ]:
plot_df.head()

In [ ]:
temp_df = plot_df.loc[:, ["IL6", "MYC", "LEP"]]
temp_df = pd.get_dummies(temp_df)
plot_lm = linear_model.LinearRegression()
plot_lm.fit(temp_df.loc[:, ["IL6", "LEP_14-16", 
                            "LEP_16-18", "LEP_18-20", 
                            "LEP_20-22"]], temp_df["MYC"])

In [ ]:
plot_lm.coef_

In [ ]:
temp_df = plot_df.loc[:, ["IL6", "MYC", "LEP"]]
temp_df = pd.get_dummies(temp_df)
plot_lm_one = linear_model.LinearRegression()
plot_lm_one.fit(temp_df.loc[:, ["IL6"]], temp_df["MYC"])

In [ ]:
plot_lm_one.coef_[0]*np.arange(8.,17.5, .25) + plot_lm.intercept_

In [ ]:
fig = plt.figure(figsize=(12,8))
ax = fig.add_subplot()

ax.scatter(plot_df["IL6"],  plot_df["MYC"], color="navy", alpha=.75)
# ax.plot(np.arange(8.,17.5, .25), 
#         np.repeat(1, len(np.arange(8.,17.5, .25))), lm_list[0].coef_[0]*np.arange(8.,17.5, .25) + lm_list[0].intercept_, 
#         color="red", lw=5, alpha=.75)

ax.plot(np.arange(8.,17.5, .25),  plot_lm_one.coef_[0]*np.arange(8.,17.5, .25) + plot_lm_one.intercept_, 
        color="red", lw=5, alpha=.75)

# ax.plot(np.arange(8.,17.5, .25), 
#         np.repeat(6, len(np.arange(8.,17.5, .25))), lm_list[5].coef_[0]*np.arange(8.,17.5, .25) + lm_list[5].intercept_, 
#         color="red", lw=5, alpha=.75)
# ax.set_title("P(MYC | IL6, LEP)", size=24)
# ax.set_xlabel("IL6", size=24)
# ax.set_ylabel("MYC", size=24)
# plt.rc('xtick', labelsize=16)
# plt.rc('ytick', labelsize=16)

# ax.set_xlim(11.5, 17.5)
# ax.set_ylim(5.5, 13.5)

In [ ]:
fig = plt.figure(figsize=(14,10), facecolor='w')
ax = fig.add_subplot(projection='3d')

ax.scatter(plot_df["IL6"], plot_df["test"], plot_df["MYC"], color="navy", alpha=.6, s=40)
# ax.plot(np.arange(8.,17.5, .25), 
#         np.repeat(1, len(np.arange(8.,17.5, .25))), lm_list[0].coef_[0]*np.arange(8.,17.5, .25) + lm_list[0].intercept_, 
#         color="red", lw=5, alpha=.75)

ax.plot(np.arange(8.,17.5, .25), 
        np.repeat(2, len(np.arange(8.,17.5, .25))), 
        plot_lm.coef_[0]*np.arange(8.,17.5, .25) + plot_lm.intercept_ + plot_lm.coef_[1], 
        color="red", lw=5, alpha=.6)

ax.plot(np.arange(8.,17.5, .25), 
        np.repeat(3, len(np.arange(8.,17.5, .25))),
          plot_lm.coef_[0]*np.arange(8.,17.5, .25) + plot_lm.intercept_ + plot_lm.coef_[2], 
        color="red", lw=5, alpha=.6)

ax.plot(np.arange(8.,17.5, .25), 
        np.repeat(4, len(np.arange(8.,17.5, .25))), 
        plot_lm.coef_[0]*np.arange(8.,17.5, .25) + plot_lm.intercept_ + plot_lm.coef_[3], 
        color="red", lw=5, alpha=.6)

ax.plot(np.arange(8.,17.5, .25), 
        np.repeat(5, len(np.arange(8.,17.5, .25))), 
        plot_lm.coef_[0]*np.arange(8.,17.5, .25) + plot_lm.intercept_ + plot_lm.coef_[4], 
        color="red", lw=5, alpha=.6)

# ax.plot(np.arange(8.,17.5, .25), 
#         np.repeat(6, len(np.arange(8.,17.5, .25))), lm_list[5].coef_[0]*np.arange(8.,17.5, .25) + lm_list[5].intercept_, 
#         color="red", lw=5, alpha=.75)
# ax.set_title("P(MYC | IL6, LEP)", size=24)
ax.set_xlabel("IL6", size=24)
ax.set_ylabel("LEP", size=24)
ax.set_zlabel("MYC", size=24)
# plt.rc('xtick', labelsize=16)
# plt.rc('ytick', labelsize=16)
ax.xaxis.labelpad=30
ax.yaxis.labelpad=30
ax.zaxis.labelpad=30
ax.set_box_aspect(aspect=None, zoom=0.75)

labels = [item.get_text() for item in ax.get_yticklabels()]
print(labels)

labels = ['15', '16', '17', '18', '19', '20', '21', '22', '']

ax.set_yticklabels(labels)

# ax.set_xlim(11.5, 17.5)
# ax.set_ylim(5.5, 13.5)

In [ ]:
import plotly.graph_objects as go
import numpy as np

x=plot_df["IL6"]
y=plot_df["test"]
z=plot_df["MYC"]

fig = go.Figure(
    data=[go.Scatter3d(z=z, x=x, y=y, opacity=0.5)])
fig.update_layout(
    title='My title', 
    autosize=True,
    width=500, 
    height=500,
)
fig.show()

In [ ]:
complete_set["LEP"].hist(bins=20)

In [ ]:
fig, ax = plt.subplots()

ax.scatter(complete_set["STAT3"], complete_set["MYC"])

In [ ]:
fig, ax = plt.subplots()

ax.scatter(complete_set["IL6"], complete_set["STAT3"])